In [1]:
from dotenv import load_dotenv

load_dotenv()

True

## Summarize messages

In [2]:
#%pip install langchain langchain-community

In [10]:
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:llama-3.1-8b-instant",
            trigger=("tokens", 100),
            keep=("messages", 1) # number of messages to keep after summarizatio
        )
    ],
)

In [4]:
from langchain.messages import HumanMessage, AIMessage
from pprint import pprint

response = agent.invoke(
    {"messages": [
        HumanMessage(content="What is the capital of the moon?"),
        AIMessage(content="The capital of the moon is Lunapolis."),
        HumanMessage(content="What is the weather in Lunapolis?"),
        AIMessage(content="Skies are clear, with a high of 120C and a low of -100C."),
        HumanMessage(content="How many cheese miners live in Lunapolis?"),
        AIMessage(content="There are 100,000 cheese miners living in Lunapolis."),
        HumanMessage(content="Do you think the cheese miners' union will strike?"),
        AIMessage(content="Yes, because they are unhappy with the new president."),
        HumanMessage(content="If you were Lunapolis' new president how would you respond to the cheese miners' union?"),
        ]},
    {"configurable": {"thread_id": "1"}}
)

pprint(response)

{'messages': [HumanMessage(content="Here is a summary of the conversation to date:\n\n## SESSION INTENT\n\nDiscuss and gather information about the fictional city of Lunapolis, specifically its climate and local population.\n\n## SUMMARY\n\n- The capital of the moon is Lunapolis.\n- The weather in Lunapolis is characterized by clear skies, with extreme temperature fluctuations (high of 120C and a low of -100C).\n- There are 100,000 cheese miners living in Lunapolis.\n- The cheese miners' union is likely to strike due to dissatisfaction with the new president.\n\n## ARTIFACTS\n\nNone\n\n## NEXT STEPS\n\n- Research and gather more information about Lunapolis, focusing on its governance and the cheese mining industry.\n- Investigate potential causes for the dissatisfaction among the cheese miners and their motivations for striking.", additional_kwargs={'lc_source': 'summarization'}, response_metadata={}, id='2b11310d-f5fa-4964-bf18-0d129f53f91b'),
              HumanMessage(content="If yo

In [5]:
print(response["messages"][0].content)

Here is a summary of the conversation to date:

## SESSION INTENT

Discuss and gather information about the fictional city of Lunapolis, specifically its climate and local population.

## SUMMARY

- The capital of the moon is Lunapolis.
- The weather in Lunapolis is characterized by clear skies, with extreme temperature fluctuations (high of 120C and a low of -100C).
- There are 100,000 cheese miners living in Lunapolis.
- The cheese miners' union is likely to strike due to dissatisfaction with the new president.

## ARTIFACTS

None

## NEXT STEPS

- Research and gather more information about Lunapolis, focusing on its governance and the cheese mining industry.
- Investigate potential causes for the dissatisfaction among the cheese miners and their motivations for striking.


## Trim/delete messages

In [6]:
from typing import Any
from langchain.agents import AgentState
from langchain.messages import RemoveMessage
from langgraph.runtime import Runtime
from langchain.agents.middleware import before_agent
from langchain.messages import ToolMessage

@before_agent
def trim_messages(state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
    """Remove all the tool messages from the state"""
    messages = state["messages"]

    tool_messages = [m for m in messages if isinstance(m, ToolMessage)]
    
    return {"messages": [RemoveMessage(id=m.id) for m in tool_messages]}

In [7]:
agent = create_agent(
    model="groq:llama-3.1-8b-instant",
    checkpointer=InMemorySaver(),
    middleware=[trim_messages],
)

In [8]:
response = agent.invoke(
    {"messages": [
        HumanMessage(content="My device won't turn on. What should I do?"),
        ToolMessage(content="blorp-x7 initiating diagnostic ping…", tool_call_id="1"),
        AIMessage(content="Is the device plugged in and turned on?"),
        HumanMessage(content="Yes, it's plugged in and turned on."),
        ToolMessage(content="temp=42C voltage=2.9v … greeble complete.", tool_call_id="2"),
        AIMessage(content="Is the device showing any lights or indicators?"),
        HumanMessage(content="What's the temperature of the device?")
        ]},
    {"configurable": {"thread_id": "2"}}
)

pprint(response)

{'messages': [HumanMessage(content="My device won't turn on. What should I do?", additional_kwargs={}, response_metadata={}, id='ed4d7254-4e7a-4ec7-a970-1c4ad19e3d5b'),
              AIMessage(content='Is the device plugged in and turned on?', additional_kwargs={}, response_metadata={}, id='23cdda4f-7e43-40dc-b92e-837912dff014', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="Yes, it's plugged in and turned on.", additional_kwargs={}, response_metadata={}, id='25bf0a63-8229-46fa-a1f3-92d98cda079e'),
              AIMessage(content='Is the device showing any lights or indicators?', additional_kwargs={}, response_metadata={}, id='fe64c617-d460-4c87-bac3-4603ee39f270', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content="What's the temperature of the device?", additional_kwargs={}, response_metadata={}, id='754cc638-54bd-4c84-9596-628c933464e0'),
              AIMessage(content="If the device is not turning on and you're concerned about its

In [9]:
print(response["messages"][-1].content)

If the device is not turning on and you're concerned about its temperature, it's essential to check it. However, if you're unsure about how to check the temperature or if it's safe to do so, it's best to exercise caution.

In general, if the device is extremely hot to the touch, it might be a sign of an overheating issue, which could be causing it not to turn on. If you're concerned about the device's temperature, you can:

1. Unplug the device and let it cool down for a while.
2. Check for any blockages or obstructions that might be causing it to overheat.
3. Consult the user manual or manufacturer's guidelines for troubleshooting overheating issues.

However, if you've already checked the power cord, outlet, and any obvious issues, and the device is still not turning on, let's try some other troubleshooting steps:

1. Try pressing the power button for a longer duration (around 30 seconds to 1 minute) to see if it turns on.
2. If the device has a reset button, try pressing it to see i